# 01 · Market Data and EDA

Purpose:

1. Download a reproducible snapshot of daily OHLCV data.
2. Validate the raw data before saving it.
3. Perform lightweight exploratory analysis without contaminating the raw dataset with engineered features.

**Important:** `market_data` remains raw OHLCV data. Any EDA-only features are created on a separate copy.

In [ ]:
!pip install -q yfinance==1.5.2

In [ ]:
from google.colab import drive
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/ai-tech-market-risk")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("yfinance:", yf.__version__)

In [ ]:
STOCKS = [
    "NVDA",
    "AMD",
    "MSFT",
    "GOOGL",
    "META",
]

BENCHMARKS = [
    "SPY",
    "QQQ",
    "SMH",
]

TICKERS = STOCKS + BENCHMARKS

START_DATE = "2018-01-01"

# yfinance treats end= as exclusive.
# This reproducible snapshot therefore requests data before 2026-08-25,
# which includes the completed 2026-08-24 session if available.
END_DATE_EXCLUSIVE = "2026-08-25"

print("Tickers:", TICKERS)
print("Start:", START_DATE)
print("End exclusive:", END_DATE_EXCLUSIVE)

## Download raw OHLCV data

Each ticker is downloaded separately so failures are easy to identify and the resulting table has a simple one-row-per-ticker-per-day structure.

In [ ]:
all_data = []

for ticker in TICKERS:
    ticker_data = yf.download(
        ticker,
        start=START_DATE,
        end=END_DATE_EXCLUSIVE,
        interval="1d",
        auto_adjust=True,
        actions=False,
        progress=False,
        threads=False,
        multi_level_index=False,
    )

    if ticker_data.empty:
        raise ValueError(f"No data downloaded for {ticker}")

    ticker_data = ticker_data.reset_index()
    ticker_data["Ticker"] = ticker

    all_data.append(ticker_data)

market_data = pd.concat(all_data, ignore_index=True)

RAW_COLUMNS = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Ticker",
]

missing_columns = set(RAW_COLUMNS) - set(market_data.columns)
if missing_columns:
    raise ValueError(f"Missing expected columns: {sorted(missing_columns)}")

market_data = (
    market_data[RAW_COLUMNS]
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
)

market_data.head()

## Validate the raw dataset

We fail loudly if duplicated ticker/date pairs, missing OHLCV values, impossible OHLC relationships, non-positive prices, or negative volume are found.

In [ ]:
summary = (
    market_data
    .groupby("Ticker")
    .agg(
        rows=("Date", "count"),
        start_date=("Date", "min"),
        end_date=("Date", "max"),
        missing_open=("Open", lambda x: x.isna().sum()),
        missing_high=("High", lambda x: x.isna().sum()),
        missing_low=("Low", lambda x: x.isna().sum()),
        missing_close=("Close", lambda x: x.isna().sum()),
        missing_volume=("Volume", lambda x: x.isna().sum()),
    )
)

summary

In [ ]:
duplicates = market_data.duplicated(
    subset=["Ticker", "Date"]
).sum()

missing_ohlcv = market_data[
    ["Open", "High", "Low", "Close", "Volume"]
].isna().sum().sum()

invalid_high = (
    market_data["High"]
    < market_data[["Open", "Close", "Low"]].max(axis=1)
).sum()

invalid_low = (
    market_data["Low"]
    > market_data[["Open", "Close", "High"]].min(axis=1)
).sum()

negative_volume = (market_data["Volume"] < 0).sum()

non_positive_prices = (
    market_data[["Open", "High", "Low", "Close"]] <= 0
).any(axis=1).sum()

observed_tickers = set(market_data["Ticker"].unique())
expected_tickers = set(TICKERS)

print("Rows:", len(market_data))
print("Columns:", market_data.shape[1])
print("Duplicate ticker/date rows:", duplicates)
print("Missing OHLCV values:", missing_ohlcv)
print("Invalid highs:", invalid_high)
print("Invalid lows:", invalid_low)
print("Negative volume:", negative_volume)
print("Rows with non-positive prices:", non_positive_prices)
print("All expected tickers present:", observed_tickers == expected_tickers)

if duplicates != 0:
    raise ValueError("Duplicate ticker/date rows detected.")

if missing_ohlcv != 0:
    raise ValueError("Missing OHLCV values detected.")

if any([
    invalid_high != 0,
    invalid_low != 0,
    negative_volume != 0,
    non_positive_prices != 0,
]):
    raise ValueError("Financial sanity checks failed.")

if observed_tickers != expected_tickers:
    raise ValueError(
        f"Ticker mismatch. Missing: {sorted(expected_tickers - observed_tickers)}"
    )

## EDA copy

`Return_1D` is useful for exploration, but it is a derived feature. We therefore calculate it on `eda_data`, not on `market_data`, so the saved raw file remains raw.

In [ ]:
eda_data = market_data.copy()

eda_data["Return_1D"] = (
    eda_data
    .groupby("Ticker")["Close"]
    .transform(
        lambda prices: prices.pct_change(fill_method=None)
    )
)

print(
    "Expected first-row return NaNs:",
    len(TICKERS)
)
print(
    "Observed Return_1D NaNs:",
    eda_data["Return_1D"].isna().sum()
)

return_summary = (
    eda_data
    .groupby("Ticker")["Return_1D"]
    .agg(["mean", "std", "min", "max"])
)

return_summary

In [ ]:
nvda = market_data[
    market_data["Ticker"] == "NVDA"
]

plt.figure(figsize=(14, 6))
plt.plot(nvda["Date"], nvda["Close"])
plt.title("NVIDIA Adjusted Closing Price")
plt.xlabel("Date")
plt.ylabel("Adjusted Price")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
prices = (
    market_data
    .pivot(index="Date", columns="Ticker", values="Close")
    .sort_index()
)

normalized_prices = prices.div(prices.iloc[0]).mul(100)

plt.figure(figsize=(14, 7))

for ticker in STOCKS:
    plt.plot(
        normalized_prices.index,
        normalized_prices[ticker],
        label=ticker,
    )

plt.title("Normalized AI / Technology Stock Performance")
plt.xlabel("Date")
plt.ylabel("Growth of 100")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Persist the raw dataset

Only the seven raw columns are saved. We then reload the file to verify that the persistent Google Drive copy is readable and has the expected shape.

In [ ]:
RAW_DATA_FILE = RAW_DATA_DIR / "market_data.csv"

market_data.to_csv(
    RAW_DATA_FILE,
    index=False,
)

print("Saved:", RAW_DATA_FILE)
print("Exists:", RAW_DATA_FILE.exists())
print("Size:", RAW_DATA_FILE.stat().st_size, "bytes")

In [ ]:
reloaded_market_data = pd.read_csv(
    RAW_DATA_FILE,
    parse_dates=["Date"],
)

expected_shape = market_data.shape
reloaded_shape = reloaded_market_data.shape

print("Original shape:", expected_shape)
print("Reloaded shape:", reloaded_shape)
print("Reload successful:", reloaded_shape == expected_shape)

if reloaded_shape != expected_shape:
    raise ValueError("Reloaded raw dataset shape does not match the saved dataset.")

if reloaded_market_data.columns.tolist() != RAW_COLUMNS:
    raise ValueError("Reloaded raw dataset schema does not match RAW_COLUMNS.")

reloaded_market_data.head()